Purpose: Find & analyze the citrate and nitrogen metabolism-related genes in the *Yucca* genomes.<br>
Author: Anna Pardo<br>
Date initiated: June 23, 2026

In [1]:
import pandas as pd
import json
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

In [2]:
# load David's supplemental table containing the lists of relevant genes
st6 = pd.read_excel("./Wickell_etal-ImpactsOfCAM-Supplementary_Tables-1.xlsx",sheet_name="Supplementary Table 6",skiprows=1,
                   usecols=["Pathway","Gene name","Gene abbreviation","Y. aloifolia gene IDs","Y. filamentosa gene IDs"])
st6.head()

,Pathway,Gene name,Gene abbreviation,Y. aloifolia gene IDs,Y. filamentosa gene IDs
0,CAM-dark,beta-carbonic anhydrase,bCA1234,Yucal.01G165600.1,YucfiPri.01G182600.1
1,CAM-dark,beta-carbonic anhydrase,bCA1234,Yucal.02G112700.1,YucfiPri.02G120100.1
2,CAM-dark,beta-carbonic anhydrase,bCA1234,NaN,YucfiPri.06G182000.1
3,CAM-dark,beta-carbonic anhydrase,bCA5,Yucal.04G001000.1,YucfiPri.04G001100.1
4,CAM-dark,beta-carbonic anhydrase,bCA5,Yucal.07G000800.1,YucfiPri.07G001300.1


In [3]:
st6["Pathway"].unique()

array(['CAM-dark', 'CAM-light', 'PhotResp', 'N-metab', 'CA-cycle'],
      dtype=object)

In [4]:
# subset to pathways of interest
path = st6[st6["Pathway"].isin(["PhotResp","N-metab","CA-cycle"])]

In [5]:
path["Gene abbreviation"].unique()

array(['PGP', 'GOX', 'GGT/GGAT', 'SHMT6,7', 'SHMT1,2', 'SHMT3', 'SHMT4,5',
       'GDC-H', 'SGT/SGAT', 'HPR', 'GLYK', 'NR', 'NiR', 'GDH', 'GS',
       'NADH-GOGAT', 'Fd-GOGAT', 'AAT', 'AAT-mito', 'ASN', 'CS4,5-mito',
       'CS1,2,3-perox', 'ACO', 'NAD-IDH1,2,3,4-reg', 'NAD-IDH5,6-cat',
       'NADP-IDH', 'OGDH-E1', 'OGDH-E2', 'SCL', 'FUM'], dtype=object)

In [6]:
# re-format with the following columns: GeneID, pathway, gene_name (family), subgenome, gene_name_unique
## first, split into two dataframes by species
yapath = path[["Pathway","Gene abbreviation","Y. aloifolia gene IDs"]].rename(columns={"Gene abbreviation":"gene_family",
                                                                                      "Y. aloifolia gene IDs":"GeneID"})
yfpath = path[["Pathway","Gene abbreviation","Y. filamentosa gene IDs"]].rename(columns={"Gene abbreviation":"gene_family",
                                                                                      "Y. filamentosa gene IDs":"GeneID"})
yfpath.head()

,Pathway,gene_family,GeneID
42,PhotResp,PGP,YucfiPri.01G330800.1
43,PhotResp,PGP,YucfiPri.22G059400.1
44,PhotResp,GOX,YucfiPri.08G083000.1
45,PhotResp,GOX,YucfiPri.09G089300.1
46,PhotResp,GOX,YucfiPri.14G056200.1


In [7]:
def add_unique_names(gname,sg,df):
    sub = df[(df["gene_family"]==gname)&(df["subgenome"]==sg)]
    x = len(sub.index)
    gnl = []
    for i in range(1,x+1):
        gnl.append(sg+"_"+gname+"_"+str(i))
    sub["gene_name_unique"] = gnl
    return sub

In [8]:
def format_species_annot(df):
    # fix gene IDs
    df2 = df.copy()
    if df2.iloc[0,2].startswith("Yucal"):
        df2["GeneID"] = df2["GeneID"].str.replace(".1$",".v2.1")
        prefix = "Ya"
    else:
        df2["GeneID"] = df2["GeneID"].str.replace(".1$",".g")
        prefix = "Yf"
        
    df2["subgenome"] = prefix
    
    return df2

In [9]:
ya_reform = format_species_annot(yapath)
yf_reform = format_species_annot(yfpath)

/tmp/ipykernel_747/1845821871.py:5: FutureWarning: The default value of regex will change from True to False in a future version.
  df2["GeneID"] = df2["GeneID"].str.replace(".1$",".v2.1")
/tmp/ipykernel_747/1845821871.py:8: FutureWarning: The default value of regex will change from True to False in a future version.
  df2["GeneID"] = df2["GeneID"].str.replace(".1$",".g")


In [10]:
ya_reform.head()

,Pathway,gene_family,GeneID,subgenome
42,PhotResp,PGP,Yucal.01G302600.v2.1,Ya
43,PhotResp,PGP,Yucal.22G052300.v2.1,Ya
44,PhotResp,GOX,Yucal.08G073500.v2.1,Ya
45,PhotResp,GOX,Yucal.09G070800.v2.1,Ya
46,PhotResp,GOX,Yucal.14G033300.v2.1,Ya


In [11]:
all_reform = pd.concat([ya_reform,yf_reform])

In [12]:
all_reform.dropna(subset="GeneID",inplace=True)

In [13]:
dfl = []
for i in all_reform["gene_family"].unique():
    for j in all_reform["subgenome"].unique():
        dfl.append(add_unique_names(i,j,all_reform))

/tmp/ipykernel_747/2717096520.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub["gene_name_unique"] = gnl
/tmp/ipykernel_747/2717096520.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub["gene_name_unique"] = gnl
/tmp/ipykernel_747/2717096520.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.

In [14]:
allpath = pd.concat(dfl)

In [15]:
allpath.tail()

,Pathway,gene_family,GeneID,subgenome,gene_name_unique
143,CA-cycle,FUM,Yucal.1Z134300.v2.1,Ya,Ya_FUM_4
144,CA-cycle,FUM,Yucal.1Z299000.v2.1,Ya,Ya_FUM_5
140,CA-cycle,FUM,YucfiPri.01G237600.g,Yf,Yf_FUM_1
141,CA-cycle,FUM,YucfiPri.02G331500.g,Yf,Yf_FUM_2
142,CA-cycle,FUM,YucfiPri.12G114000.g,Yf,Yf_FUM_3


In [45]:
allpath.to_csv("./photresp_N_citrate_genes_Yucca.csv",sep=",",header=True,index=False)

In [16]:
# June 30
# Problem: in this table, Yf genes are all from v2!
# I don't remember how exactly, but: I have a version of David's table with v3 gene annotations, here:
yfv3_pathways = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/degs_downstream/Yfv3_genesofint_info.txt",
                           sep="\t",header="infer")
yfv3_pathways.head()

,Yf_genes,Orthogroup,Pathway,gene_name,gene_abbr
0,YufilH1007286m.g,OG0001252,CA-cycle,2-Oxoglutarate dehydrogenase (E1 subunit),OGDH-E1
1,YufilH1007287m.g,OG0001252,CA-cycle,2-Oxoglutarate dehydrogenase (E1 subunit),OGDH-E1
2,YufilH1007288m.g,OG0001252,CA-cycle,2-Oxoglutarate dehydrogenase (E1 subunit),OGDH-E1
3,YufilH1009437m.g,OG0001252,CA-cycle,2-Oxoglutarate dehydrogenase (E1 subunit),OGDH-E1
4,YufilH1009438m.g,OG0001252,CA-cycle,2-Oxoglutarate dehydrogenase (E1 subunit),OGDH-E1


In [18]:
# overwrite yf_reform
yf_reform = yfv3_pathways[["Pathway","gene_abbr","Yf_genes"]].rename(columns={"gene_abbr":"gene_family","Yf_genes":"GeneID"})
yf_reform["subgenome"] = "Yf"
yf_reform.head()

,Pathway,gene_family,GeneID,subgenome
0,CA-cycle,OGDH-E1,YufilH1007286m.g,Yf
1,CA-cycle,OGDH-E1,YufilH1007287m.g,Yf
2,CA-cycle,OGDH-E1,YufilH1007288m.g,Yf
3,CA-cycle,OGDH-E1,YufilH1009437m.g,Yf
4,CA-cycle,OGDH-E1,YufilH1009438m.g,Yf


In [19]:
# redo formatting
all_reform = pd.concat([ya_reform,yf_reform])
all_reform.head()

,Pathway,gene_family,GeneID,subgenome
42,PhotResp,PGP,Yucal.01G302600.v2.1,Ya
43,PhotResp,PGP,Yucal.22G052300.v2.1,Ya
44,PhotResp,GOX,Yucal.08G073500.v2.1,Ya
45,PhotResp,GOX,Yucal.09G070800.v2.1,Ya
46,PhotResp,GOX,Yucal.14G033300.v2.1,Ya


In [20]:
all_reform.dropna(subset="GeneID",inplace=True)

In [21]:
dfl = []
for i in all_reform["gene_family"].unique():
    for j in all_reform["subgenome"].unique():
        dfl.append(add_unique_names(i,j,all_reform))

/tmp/ipykernel_747/2717096520.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub["gene_name_unique"] = gnl
/tmp/ipykernel_747/2717096520.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub["gene_name_unique"] = gnl
/tmp/ipykernel_747/2717096520.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.

In [22]:
allpath = pd.concat(dfl)

In [23]:
allpath.tail()

,Pathway,gene_family,GeneID,subgenome,gene_name_unique
195,N-metab,AAT-cyto,YufilH1038132m.g,Yf,Yf_AAT-cyto_2
196,N-metab,AAT-plastid,YufilH1088791m.g,Yf,Yf_AAT-plastid_1
197,N-metab,AAT-plastid,YufilH1088792m.g,Yf,Yf_AAT-plastid_2
198,N-metab,AAT-plastid,YufilH1088793m.g,Yf,Yf_AAT-plastid_3
199,N-metab,AAT-plastid,YufilH1094409m.g,Yf,Yf_AAT-plastid_4


In [24]:
# overwrite file
allpath.to_csv("./photresp_N_citrate_genes_Yucca.csv",sep=",",header=True,index=False)